# 🧩 Mini-Lab: Create a Virtual Environment

**Module: Escape the Notebook: Environment, Repo, Git** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** why isolated environments are essential for reproducibility
2. **Create** a virtual environment using Python's built-in `venv` module
3. **Activate** the environment and **confirm** which Python interpreter is active

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Virtual Environments | A virtual environment is an isolated workspace that keeps a project’s Python packages and settings separate, so installs and upgrades do not break other projects. |
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. |


## Why Isolation Matters

Recall from **mini-repo-reproducibility-audit**: a project is reproducible when a stranger can clone it and run it using only what's in the repo.

But what happens when two projects need **different versions** of the same package?

| Scenario | System Python | With Virtual Environments |
|----------|--------------|---------------------------|
| Project A needs `requests==2.28` | ❌ Conflicts with Project B | ✅ Each project has its own copy |
| Project B needs `requests==2.31` | ❌ Only one can win | ✅ No conflict |
| You upgrade a package globally | ❌ Might break other projects | ✅ Only affects the active env |

A **virtual environment** is a self-contained directory that has its own Python interpreter and its own `site-packages`. Packages installed inside it are invisible to other projects — and vice versa.

> **Key insight:** Without isolation, installing a package for one project can silently break another. Virtual environments make each project's dependencies independent and predictable.

## Step 1 — See Where Your System Python Lives

Before creating an environment, let's record which Python we're currently using. This is the **system** (or base) interpreter — the one that every project would share if we didn't isolate.

In [1]:
import sys

print("Python interpreter:", sys.executable)
print("Python version:    ", sys.version.split()[0])
print("Prefix (base env): ", sys.prefix)

Python interpreter: C:\Users\molak\.virtualenvs\text_llm_agent\Scripts\python.exe
Python version:     3.12.10
Prefix (base env):  C:\Users\molak\.virtualenvs\text_llm_agent


Notice the `sys.prefix` — it points to your system or base environment. After we create and activate a virtual environment, this value will change to point to the new, isolated directory.

## Step 2 — Create & Activate a Virtual Environment (Terminal)

Virtual environments are created and activated **in the terminal**, not inside a notebook. Open a terminal and navigate to a project folder (or create one), then follow the steps below.

### 2a — Create the environment

Run this command to create a virtual environment named `.venv` inside your project folder:

```bash
python -m venv .venv
```

> **Why `.venv`?** The leading dot hides it on macOS/Linux, and the name is the most common convention. Many tools (VS Code, PyCharm) auto-detect it.

This creates a `.venv/` directory containing:
- A copy of (or symlink to) the Python interpreter
- An empty `site-packages/` — no third-party packages yet
- Activation scripts for your shell

### 2b — Activate the environment

The activation command differs by operating system:

| OS | Shell | Command |
|----|-------|---------|
| macOS / Linux | bash / zsh | `source .venv/bin/activate` |
| Windows | Command Prompt | `.venv\Scripts\activate` |
| Windows | PowerShell | `.venv\Scripts\Activate.ps1` |

After activation, your terminal prompt changes — you'll see `(.venv)` at the beginning, confirming you're inside the isolated environment.

### 2c — Confirm the active interpreter

While the environment is activated, run:

```bash
python -c "import sys; print(sys.executable)"
```

The path should now point **inside** your `.venv/` directory — not your system Python. This proves isolation is active.

## Step 3 — Inspect the Difference: Before vs After

Let's simulate what changes when you activate a virtual environment, so you can see the contrast clearly — even from inside this notebook.

In [2]:
import os
from pathlib import Path
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Simulate the before/after paths for demonstration
# (Actual paths depend on your OS; these illustrate the pattern)

is_windows = os.name == "nt"

if is_windows:
    system_python = r"C:\Python310\python.exe"
    venv_python   = r"C:\my-project\.venv\Scripts\python.exe"
    venv_packages = r"C:\my-project\.venv\Lib\site-packages"
else:
    system_python = "/usr/bin/python3"
    venv_python   = "/home/user/my-project/.venv/bin/python3"
    venv_packages = "/home/user/my-project/.venv/lib/python3.10/site-packages"

comparison = f"""### Before Activation (System Python)

| Property | Value |
|----------|-------|
| `sys.executable` | `{system_python}` |
| `pip install X` goes to | System-wide `site-packages` (shared by all projects) |
| Risk | Installing a package here can break other projects |

### After Activation (Virtual Environment)

| Property | Value |
|----------|-------|
| `sys.executable` | `{venv_python}` |
| `pip install X` goes to | `{venv_packages}` (project-only) |
| Risk | None — changes are isolated to this project |
"""

md(comparison)

### Before Activation (System Python)

| Property | Value |
|----------|-------|
| `sys.executable` | `C:\Python310\python.exe` |
| `pip install X` goes to | System-wide `site-packages` (shared by all projects) |
| Risk | Installing a package here can break other projects |

### After Activation (Virtual Environment)

| Property | Value |
|----------|-------|
| `sys.executable` | `C:\my-project\.venv\Scripts\python.exe` |
| `pip install X` goes to | `C:\my-project\.venv\Lib\site-packages` (project-only) |
| Risk | None — changes are isolated to this project |


## Step 4 — Verify Isolation: Empty `site-packages`

A fresh virtual environment has **almost no packages** — just `pip` and `setuptools`. This is the whole point: you start clean and install only what the project needs.

With the environment activated in your terminal, run:

```bash
pip list
```

You should see something like:

```
Package    Version
---------- -------
pip        23.x.x
setuptools 68.x.x
```

Compare that to running `pip list` **outside** the venv — the system will have dozens (or hundreds) of packages. The contrast makes the isolation concrete.

## Step 5 — Deactivate

When you're done working in the project, deactivate the environment:

```bash
deactivate
```

The `(.venv)` prefix disappears from your prompt, and `python` points back to the system interpreter. The virtual environment still exists on disk — you can reactivate it anytime.

## Quick-Reference Cheat Sheet

In [3]:
cheat_sheet = """
### Virtual Environment Commands

| Action | Command |
|--------|---------|
| **Create** | `python -m venv .venv` |
| **Activate (macOS/Linux)** | `source .venv/bin/activate` |
| **Activate (Windows CMD)** | `.venv\\Scripts\\activate` |
| **Activate (Windows PS)** | `.venv\\Scripts\\Activate.ps1` |
| **Confirm active interpreter** | `python -c "import sys; print(sys.executable)"` |
| **Check installed packages** | `pip list` |
| **Deactivate** | `deactivate` |
| **Delete environment** | Delete the `.venv/` folder |

> **Tip:** Always add `.venv/` to your `.gitignore` — the environment is local and should never be committed.
"""

md(cheat_sheet)


### Virtual Environment Commands

| Action | Command |
|--------|---------|
| **Create** | `python -m venv .venv` |
| **Activate (macOS/Linux)** | `source .venv/bin/activate` |
| **Activate (Windows CMD)** | `.venv\Scripts\activate` |
| **Activate (Windows PS)** | `.venv\Scripts\Activate.ps1` |
| **Confirm active interpreter** | `python -c "import sys; print(sys.executable)"` |
| **Check installed packages** | `pip list` |
| **Deactivate** | `deactivate` |
| **Delete environment** | Delete the `.venv/` folder |

> **Tip:** Always add `.venv/` to your `.gitignore` — the environment is local and should never be committed.


## How This Connects to Reproducibility

In **mini-repo-reproducibility-audit**, we saw that a reproducible repo needs a `requirements.txt` with pinned versions. But pinned versions only help if each project installs into its **own** isolated environment.

The workflow becomes:

1. **Create** a virtual environment → `python -m venv .venv`
2. **Activate** it → `source .venv/bin/activate`
3. **Install** pinned dependencies → `pip install -r requirements.txt`
4. **Run** the project → everything uses the isolated interpreter

This is exactly what the "stranger cloning your repo" will do. Without step 1, step 3 pollutes their system Python and risks breaking their other projects.

## 🎯 Summary

### Key Takeaways

1. **Virtual environments isolate dependencies** — each project gets its own Python interpreter and `site-packages`, preventing conflicts between projects
2. **`python -m venv .venv` is all you need** — Python's built-in `venv` module creates a lightweight isolated environment with no extra tools required
3. **`sys.executable` is the proof** — after activation, checking the interpreter path confirms you're running inside the isolated environment, not the system Python
4. **Isolation enables reproducibility** — virtual environments are the foundation that makes `pip install -r requirements.txt` safe and predictable for anyone cloning your repo